# 04 — Model comparison

> This system predicts glucose values for research and analysis. It is not a medical device and must not be used to determine insulin doses or make treatment decisions.

See also `reports/model_comparison.md`. The test period is used only in notebook 05.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config, db, train
settings, mcfg = config.settings(), config.models()
duck = db.duck(read_only=True)
q = lambda sql: duck.execute(sql).df()

In [ ]:
q("""SELECT model, target_type, feature_set, horizon, round(mae, 3) mae, round(rmse, 3) rmse, round(r2, 3) r2
     FROM experiments WHERE kind = 'cross_validation' ORDER BY horizon, mae""")

In [ ]:
cvres = q("SELECT * FROM cv_results WHERE fold = 'pooled'")
piv = cvres[cvres.feature_set.isin(['-', 'glucose_insulin', 'sequence'])].pivot_table(index=['model', 'target_type'], columns='horizon', values='mae')
piv.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for (m, t), r in piv.iterrows():
    ax.plot(r.index, r.values, marker='o', label=f'{m} ({t})')
ax.set_xlabel('horizon (min)'); ax.set_ylabel('MAE (mmol/L)'); ax.legend(fontsize=7)